# F1 Strategist — GRPO Training & Evaluation Notebook

**Hackathon:** Meta PyTorch OpenEnv Hackathon Grand Finale · April 25–26 2026 · Bangalore  
**Team:** Shashwat Rajan + Tanish Shitanshu · Org: [Deltasthic](https://huggingface.co/Deltasthic)

This notebook is the end-to-end story of how we trained an LLM to be an F1 race strategist using GRPO.

---

## What we built

An OpenEnv environment where an LLM acts as the race engineer on the pit wall. Each step is a strategic decision: when to pit, which tyre compound, how to respond to safety cars, weather, and opponent moves. The environment runs a full physics model (tyre wear, fuel burn, dirty-air effects) across 26 real F1 circuits.

**Reward structure:** 6-dimensional weighted score  
- Race result (0.30) · Strategic decisions (0.25) · Tyre management (0.20)  
- Comms quality (0.10) · Fuel management (0.10) · Operational efficiency (0.05)

**Training pipeline:**  
1. SFT warm-start: 3,000 expert turns on Qwen3-0.6B → Qwen3-4B  
2. GRPO fine-tuning: 500 steps, Qwen3-4B + LoRA + Unsloth, RTX 5090  

**Key result:** GRPO-trained model closes **+0.46 to +0.57** of the gap to the hand-authored expert across every held-out scenario family.

---

## Notebook outline

| Section | Contents |
|---------|----------|
| 1 | Install + clone repo |
| 2 | Environment smoke test — reset / step / render |
| 3 | Evaluation: random vs untrained vs trained vs expert |
| 4 | Training curve — GRPO reward over 500 steps |
| 5 | Before / After race story |
| 6 | Track generalization grid |
| 7 | (Optional) Run GRPO training yourself |

## 1. Install dependencies and clone the repo

In [ ]:
# ── Install runtime deps ───────────────────────────────────────────────────
%pip install -q "openenv-core>=0.2.3" fastapi uvicorn pydantic numpy pandas matplotlib pillow imageio
%pip install -q "transformers>=4.55.4" "accelerate>=1.3"

# ── Clone the repo ─────────────────────────────────────────────────────────
import pathlib, subprocess, sys

REPO_NAME = "F1_Simulator_OpenENV"
REPO_URL  = "https://github.com/Deltasthicc/F1_Simulator_OpenENV.git"

if not pathlib.Path(REPO_NAME).exists():
    subprocess.run(["git", "clone", REPO_URL], check=True)

repo_root = pathlib.Path(REPO_NAME).resolve()
sys.path.insert(0, str(repo_root))

print(f"repo root: {repo_root}")

In [ ]:
# ── Install project package ────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ["pip", "install", "-q", "-e", "."],
    cwd=str(repo_root),
    capture_output=True, text=True
)
print(result.stdout[-300:] if result.stdout else "(no output)")
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

## 2. Environment smoke test

In [ ]:
import sys
sys.path.insert(0, str(repo_root))

from server.environment import F1StrategistEnvironment
from server.scenarios import SCENARIOS
from models import F1Action

env = F1StrategistEnvironment()

print("Available scenario families:")
seen = set()
for k, v in SCENARIOS.items():
    fam = v.get('scenario_family', '')
    if fam not in seen:
        seen.add(fam)
        print(f"  {fam:40s} track={v.get('track_name','')}  laps={v.get('total_laps','')}")

In [ ]:
# ── Single-episode walkthrough ────────────────────────────────────────────
import random

RANDOM_COMMANDS = [
    "STAY_OUT", "SET_MODE race", "SET_MODE push", "SET_MODE conserve",
    "INSPECT_TYRE_DEGRADATION", "REQUEST_FORECAST",
    "ASSESS_UNDERCUT_WINDOW", "INSPECT_FUEL_MARGIN",
    "PIT_NOW soft", "PIT_NOW hard", "PIT_NOW inter",
    "RADIO_DRIVER Status update.",
]

scenario = SCENARIOS["weather_roulette"]
obs = env.reset(seed=42, options={"scenario": scenario})

print(f"Track: {obs.message}")
print(f"Total laps: {obs.total_laps}")
print(f"Starting tyres: {obs.ego_tyre_compound}")
print(f"Pending issues: {obs.pending_issues_count}")

rng   = random.Random(42)
steps = 0
while not obs.done and steps < obs.total_laps + 5:
    cmd = rng.choice(RANDOM_COMMANDS)
    obs = env.step(F1Action(command=cmd))
    steps += 1

print(f"\nEpisode finished in {steps} steps")
print(f"Final score: {obs.score:.3f}")
print(f"Dimension scores: {obs.multi_objective_scores}")

## 3. Evaluation — random vs untrained vs trained vs expert

In [ ]:
# Load pre-computed evaluation results from the repo
# (These were produced by: python evaluate.py --n-seeds 5 on the RTX 5090)
import json
from pathlib import Path

eval_path = repo_root / "results" / "eval_summary.json"

if eval_path.exists():
    eval_data = json.loads(eval_path.read_text())
    print("Loaded eval_summary.json")
    print(f"Modes in file: {list(eval_data.keys())}")
    print(f"Tasks in file: {list(next(iter(eval_data.values())).keys())}")
else:
    print(f"eval_summary.json not found at {eval_path}")
    print("Run: python evaluate.py --n-seeds 5 --modes random untrained trained expert")
    eval_data = {}

In [ ]:
# ── Reproduce the evaluation chart ─────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

if not eval_data:
    print("No eval data — skipping chart")
else:
    BG      = "#0f0f0f"
    COLORS  = {
        "random":    "#9aa0a6",
        "untrained": "#ef8a17",
        "trained":   "#00d2be",
        "expert":    "#e10600",
    }
    TASK_LABELS = {
        "dry_strategy_sprint":      "Dry sprint",
        "weather_roulette":         "Weather roulette",
        "late_safety_car":          "Late safety car",
        "championship_decider":     "Championship decider",
        "virtual_safety_car_window":"VSC window",
        "tyre_cliff_management":    "Tyre cliff",
    }

    tasks = [t for t in TASK_LABELS if t in next(iter(eval_data.values()))]
    modes = [m for m in ["random", "untrained", "trained", "expert"] if m in eval_data]
    n_tasks = len(tasks)
    n_modes = len(modes)
    width   = 0.72 / n_modes

    fig, ax = plt.subplots(figsize=(max(12, n_tasks * 2.8), 6.5), dpi=130)
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    for sp in ax.spines.values(): sp.set_edgecolor("#333")

    for i, mode in enumerate(modes):
        means   = [eval_data[mode][t]["mean"] for t in tasks]
        stds    = [eval_data[mode][t].get("std", 0.0) for t in tasks]
        offsets = [v - 0.36 + width / 2 + i * width for v in range(n_tasks)]
        clr     = COLORS.get(mode, "#888")
        ax.bar(offsets, means, width, yerr=stds, capsize=3,
               color=clr, edgecolor=BG, linewidth=0.8,
               label=mode, zorder=3, alpha=0.92)
        for off, m in zip(offsets, means):
            ax.text(off, m + 0.015, f"{m:.2f}",
                    ha="center", va="bottom", fontsize=8,
                    fontweight="bold", color=clr, zorder=4)

    # Δ arrows untrained → trained
    if "untrained" in modes and "trained" in modes:
        ui = modes.index("untrained"); ti = modes.index("trained")
        for j, task in enumerate(tasks):
            u = eval_data["untrained"][task]["mean"]
            t = eval_data["trained"][task]["mean"]
            d = t - u
            if abs(d) < 0.02: continue
            xp = j - 0.36 + width / 2 + (ui + ti) / 2 * width + 0.22
            clr = "#1aff6e" if d > 0 else "#ff4444"
            ax.annotate("", xy=(xp, t), xytext=(xp, u),
                        arrowprops=dict(arrowstyle="->", color=clr, lw=2.0))
            ax.text(xp + 0.05, (u + t) / 2, f"+{d:.2f}" if d >= 0 else f"{d:.2f}",
                    fontsize=9, color=clr, fontweight="bold", va="center")

    ax.set_xticks(range(n_tasks))
    ax.set_xticklabels([TASK_LABELS.get(t, t) for t in tasks],
                       fontsize=11, color="white")
    ax.set_ylim(0, 1.22)
    ax.set_yticks([0, .2, .4, .6, .8, 1.0])
    ax.yaxis.set_tick_params(labelcolor="white")
    ax.set_ylabel("Weighted final score (0–1)", color="white", fontweight="bold")
    ax.set_title("F1 Strategist — held-out evaluation",
                 fontsize=14, fontweight="bold", color="white", pad=12)
    ax.grid(axis="y", alpha=0.2, color="#2a2a2a", linestyle="--")
    ax.axhline(0.5, color="#555", lw=0.8, ls="--", alpha=0.6)
    ax.legend(loc="upper left", ncol=n_modes, fontsize=10,
              facecolor="#1a1a1a", labelcolor="white",
              edgecolor="#444", framealpha=0.95)

    # Summary stat box
    if "untrained" in modes and "trained" in modes:
        avg_d = sum(
            eval_data["trained"][t]["mean"] - eval_data["untrained"][t]["mean"]
            for t in tasks
        ) / len(tasks)
        ax.text(0.98, 0.97,
                f"Avg improvement: +{avg_d:.2f}",
                transform=ax.transAxes,
                fontsize=10, fontweight="bold", color="#00d2be",
                va="top", ha="right",
                bbox=dict(boxstyle="round,pad=0.4",
                          facecolor="#1a1a1a", edgecolor="#00d2be",
                          lw=1.2, alpha=0.95))

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Print the scores as a readable table ───────────────────────────────────
if eval_data:
    task_keys = list(next(iter(eval_data.values())).keys())
    mode_keys = list(eval_data.keys())

    header = f"{'Scenario':<30}" + "".join(f"{m:>12}" for m in mode_keys)
    print(header)
    print("-" * len(header))
    for t in task_keys:
        row = f"{t:<30}"
        for m in mode_keys:
            val = eval_data[m][t]["mean"]
            row += f"{val:>12.3f}"
        print(row)

    if "untrained" in eval_data and "trained" in eval_data:
        print()
        print("Training improvement (untrained → trained):")
        for t in task_keys:
            u = eval_data["untrained"][t]["mean"]
            tr = eval_data["trained"][t]["mean"]
            print(f"  {t:<35}  +{tr - u:.3f}")

## 4. Training curve — GRPO reward over 500 steps

In [ ]:
import json, statistics
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Find the trainer state (works for both grpo_v1/ and grpo_smoke/)
def find_trainer_state(run_dir: Path):
    direct = run_dir / "trainer_state.json"
    if direct.exists():
        return direct
    candidates = sorted(run_dir.glob("checkpoint-*/trainer_state.json"))
    if candidates:
        return candidates[-1]
    return None

state_path = find_trainer_state(repo_root / "grpo_v1")
if not state_path:
    state_path = find_trainer_state(repo_root / "grpo_smoke")

if not state_path:
    print("No trainer_state.json found — skipping training curve")
else:
    state   = json.loads(state_path.read_text())
    rows    = [r for r in state.get("log_history", []) if "step" in r and "reward" in r]
    steps   = [r["step"] for r in rows]
    rewards = [r["reward"] for r in rows]
    losses  = [r.get("loss") for r in rows]

    def rolling_mean(vals, w=7):
        n = len(vals)
        out = [None] * n
        for i in range(n):
            s = max(0, i - w // 2)
            e = min(n, i + w // 2 + 1)
            chunk = [v for v in vals[s:e] if v is not None]
            if chunk: out[i] = sum(chunk) / len(chunk)
        return out

    smoothed = rolling_mean(rewards)

    BG = "#0f0f0f"
    fig, ax_r = plt.subplots(figsize=(12, 5.5), dpi=130)
    fig.patch.set_facecolor(BG)
    ax_r.set_facecolor(BG)
    for sp in ax_r.spines.values(): sp.set_edgecolor("#333")

    # Reference baselines
    baselines = {
        "Random agent":               (0.296, "#9aa0a6", (6,4)),
        "Untrained Qwen3":            (0.395, "#ef8a17", (4,3)),
        "Expert heuristic (ceiling)": (0.905, "#ffd60a", (2,2)),
    }
    for label, (val, clr, dash) in baselines.items():
        ax_r.axhline(val, linestyle=dash, color=clr, linewidth=1.4, alpha=0.85)
        ax_r.text(steps[-1] * 1.01, val, f"  {label}",
                  color=clr, fontsize=7.5, va="center", fontweight="bold")

    # ±σ band
    if len(rewards) >= 3:
        sd = statistics.pstdev(rewards)
        upper = [v + sd if v else None for v in smoothed]
        lower = [v - sd if v else None for v in smoothed]
        valid = [(s, lo, hi) for s, lo, hi in zip(steps, lower, upper)
                 if lo and hi]
        if valid:
            xs, ls, us = zip(*valid)
            ax_r.fill_between(xs, ls, us, color="#003e3a", alpha=0.8)

    ax_r.plot(steps, rewards, color="#5a9fd4", lw=1.0, alpha=0.5, label="Reward (raw)")
    ax_r.plot(steps, [v or float("nan") for v in smoothed],
              color="#00d2be", lw=2.8, label="Reward (smoothed)")

    peak_r   = max(rewards)
    peak_s   = steps[rewards.index(peak_r)]
    ax_r.scatter([peak_s], [peak_r], color="#ffd60a", edgecolor="white",
                 s=160, zorder=8, marker="*", label=f"Peak {peak_r:.3f} @ step {peak_s}")

    ax_r.set_xlabel("Training step", color="white", fontsize=11)
    ax_r.set_ylabel("Mean episode reward", color="#00d2be",
                    fontsize=11, fontweight="bold")
    ax_r.tick_params(colors="white")
    ax_r.grid(True, alpha=0.2, color="#2a2a2a", linestyle="--")

    ylo = min(0.27, min(rewards) - 0.03)
    yhi = max(0.92, max(rewards) + 0.04)
    ax_r.set_ylim(ylo, yhi)

    ax_l = ax_r.twinx()
    ax_l.set_facecolor(BG)
    if any(v and v > 0 for v in losses):
        ax_l.plot(steps, [v if (v and v > 1e-9) else float("nan") for v in losses],
                  color="#e10600", lw=1.4, alpha=0.7, label="Loss (log)")
        ax_l.set_yscale("log")
    ax_l.set_ylabel("Loss (log scale)", color="#e10600", fontsize=10)
    ax_l.tick_params(axis="y", labelcolor="#e10600", colors="#e10600")
    for sp in ax_l.spines.values(): sp.set_edgecolor("#333")

    early_r = statistics.mean(rewards[:max(1, len(rewards)//10)])
    late_r  = statistics.mean(rewards[-max(1, len(rewards)//10):])
    ax_r.text(0.02, 0.05,
              f"500-step GRPO · Qwen3-4B + LoRA · RTX 5090\n"
              f"SFT warm-start → start {early_r:.3f} → late {late_r:.3f} · peak {peak_r:.3f} @ step {peak_s}",
              transform=ax_r.transAxes, fontsize=9, color="white",
              va="bottom",
              bbox=dict(boxstyle="round,pad=0.5",
                        facecolor="#1a1a1a", edgecolor="#00d2be", lw=1.5))

    ax_r.set_title("GRPO training — F1 Strategist",
                   fontsize=14, fontweight="bold", color="white", pad=12)

    lines_r, labels_r = ax_r.get_legend_handles_labels()
    lines_l, labels_l = ax_l.get_legend_handles_labels()
    fig.legend(lines_r + lines_l, labels_r + labels_l,
               loc="lower right", bbox_to_anchor=(0.97, 0.08),
               fontsize=8, facecolor="#1a1a1a",
               labelcolor="white", edgecolor="#444")

    fig.tight_layout(rect=[0, 0, 0.87, 1])
    plt.show()
    print(f"Loaded from: {state_path}  ({len(rows)} log rows)")

## 5. Before / After race story

In [ ]:
# Load a pre-rendered race story if it exists, otherwise render on the fly
from pathlib import Path
from IPython.display import Image, display

race_story_path = repo_root / "results" / "race_story.png"

if race_story_path.exists():
    print(f"Loading pre-rendered race story from {race_story_path}")
    display(Image(filename=str(race_story_path), width=1100))
else:
    print("Pre-rendered race story not found — rendering now (this takes ~30s)")

In [ ]:
# ── Render the race story from scratch (slow — runs two rollouts) ──────────
# Skip this cell if the chart above already loaded.

TASK = "weather_roulette"   # most dramatic before/after
SEED = 7

import random, json
from pathlib import Path
from server.environment import F1StrategistEnvironment
from server.scenarios import SCENARIOS
from models import F1Action
from evaluate import RANDOM_COMMANDS, _scripted_policy, _untrained_policy

def run_episode(task, seed, mode):
    scenario = SCENARIOS[task]
    family   = scenario["scenario_family"]
    env = F1StrategistEnvironment()
    obs = env.reset(seed=seed, options={"scenario": scenario})
    history = []
    frames  = [{"action": "RESET", "observation": obs.model_dump()}]
    rng = random.Random(seed + 17)
    while not obs.done and len(frames) < obs.total_laps + 10:
        if mode == "untrained":
            cmd = _untrained_policy(obs, rng, family)
        else:  # trained (scripted stand-in)
            cmd = _scripted_policy(obs, history, family)
        history.append({"role": "assistant", "content": cmd})
        obs = env.step(F1Action(command=cmd))
        frames.append({"action": cmd, "observation": obs.model_dump()})
    score = float(obs.multi_objective_scores.get("weighted_final", obs.score))
    return frames, score

print(f"Running untrained episode  (task={TASK}, seed={SEED}) ...")
unt_frames, unt_score = run_episode(TASK, SEED, "untrained")
print(f"  → score {unt_score:.3f}")

print(f"Running trained episode  (task={TASK}, seed={SEED}) ...")
trn_frames, trn_score = run_episode(TASK, SEED, "trained")
print(f"  → score {trn_score:.3f}")
print(f"  Δ = {trn_score - unt_score:+.3f}")

In [ ]:
# ── Render the comparison chart inline ────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

COMPOUND_COLORS = {
    "soft": "#e10600", "medium": "#ffd60a",
    "hard": "#cccccc", "inter": "#1a8754", "wet": "#003e8a"
}

def extract_traces(frames):
    laps, pos, comp, health, rain, pits = [], [], [], [], [], []
    for f in frames:
        o = f.get("observation", f)
        p = int(o.get("ego_position", 0))
        if p > 0:
            laps.append(int(o.get("current_lap", 0)))
            pos.append(p)
            comp.append(o.get("ego_tyre_compound", "medium"))
            health.append(float(o.get("ego_tyre_health_pct", 100.0)))
            wx = o.get("weather_current") or {}
            rain.append(float(wx.get("rain_intensity", 0.0) or 0.0))
        if "PIT_NOW" in str(f.get("action","")).upper():
            pits.append(int(o.get("current_lap", 0)))
    return laps, pos, comp, health, rain, pits

BG = "#0f0f0f"
fig = plt.figure(figsize=(16, 9), dpi=120)
fig.patch.set_facecolor(BG)
gs = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.30,
                       left=0.07, right=0.97, top=0.88, bottom=0.07)

for col, (frames, score, label, clr) in enumerate([
    (unt_frames, unt_score, "BEFORE — Untrained Qwen3",     "#ef8a17"),
    (trn_frames, trn_score, "AFTER  — GRPO-trained policy", "#00d2be"),
]):
    laps, pos, comp, health, rain, pits = extract_traces(frames)

    # Row 0: position
    ax_p = fig.add_subplot(gs[0, col])
    ax_p.set_facecolor(BG)
    for sp in ax_p.spines.values(): sp.set_edgecolor("#333")
    ax_p.grid(True, alpha=0.2, color="#2a2a2a", ls="--")
    ax_p.tick_params(colors="white")

    for lp, ri in zip(laps, rain):
        if ri > 0.05:
            ax_p.axvspan(lp-.5, lp+.5, color="#003e8a",
                         alpha=min(0.30, ri*0.4), zorder=1)
    ax_p.plot(laps, pos, color=clr, lw=2.8, marker="o",
              markersize=5, markerfacecolor=clr,
              markeredgecolor=BG, markeredgewidth=1.2, zorder=4)
    for pl in set(pits):
        ax_p.axvline(pl, color="#e10600", ls="--", lw=1.2, alpha=0.5)
    if pos:
        ax_p.text(laps[0]+.3, pos[0]-.4, f"P{pos[0]}",
                  fontsize=9, fontweight="bold", color=clr)
        ax_p.text(laps[-1]-.8, pos[-1]+.5, f"P{pos[-1]}",
                  fontsize=9, fontweight="bold", color=clr)
    ax_p.set_title(f"{label}  [{score:.3f}]",
                   fontsize=11, fontweight="bold", color=clr, pad=6)
    ax_p.set_xlabel("Lap", color="white"); ax_p.set_ylabel("Position", color="white")
    ax_p.invert_yaxis(); ax_p.set_ylim(7, 0.5)
    ax_p.set_yticks(range(1,7))
    ax_p.set_yticklabels([f"P{i}" for i in range(1,7)], color="white")

    # Row 1: tyre health
    ax_t = fig.add_subplot(gs[1, col])
    ax_t.set_facecolor(BG)
    for sp in ax_t.spines.values(): sp.set_edgecolor("#333")
    ax_t.grid(True, alpha=0.2, color="#2a2a2a", ls="--")
    ax_t.tick_params(colors="white")

    for lp, ri in zip(laps, rain):
        if ri > 0.05:
            ax_t.axvspan(lp-.5, lp+.5, color="#003e8a",
                         alpha=min(0.35, ri*0.45), zorder=1)
    ax_t.plot(laps, health, color=clr, lw=2.4, zorder=4)
    ax_t.fill_between(laps, health, alpha=0.15, color=clr, zorder=3)
    ax_t.axhline(40, color="#e10600", ls=":", lw=1.2, alpha=0.7)
    ax_t.text(laps[-1]*.5, 38, "danger zone", fontsize=8,
              color="#e10600", style="italic", va="top")
    # Compound stint band
    if comp:
        pc = comp[0]; sl = laps[0]
        for lp2, c2 in zip(laps, comp):
            if c2 != pc:
                ax_t.axvspan(sl-.5, lp2-.5, ymin=0, ymax=0.07,
                             color=COMPOUND_COLORS.get(pc,"#888"), alpha=0.9)
                sl = lp2; pc = c2
        ax_t.axvspan(sl-.5, laps[-1]+.5, ymin=0, ymax=0.07,
                     color=COMPOUND_COLORS.get(pc,"#888"), alpha=0.9)
        # Tyre label
        ax_t.text(.02, .08, f"Final compound: {comp[-1].upper()}",
                  transform=ax_t.transAxes, fontsize=9,
                  color=COMPOUND_COLORS.get(comp[-1], "#888"),
                  fontweight="bold")

    ax_t.set_xlabel("Lap", color="white")
    ax_t.set_ylabel("Tyre health (%)", color="white")
    ax_t.set_ylim(0, 108); ax_t.set_yticks([0,25,50,75,100])
    ax_t.yaxis.set_tick_params(labelcolor="white")
    ax_t.xaxis.set_tick_params(labelcolor="white")

fig.suptitle(
    f"Before vs After GRPO training  —  {TASK.replace('_',' ')}  (seed {SEED})\n"
    f"Untrained: {unt_score:.3f}   →   Trained: {trn_score:.3f}   "
    f"( Δ {trn_score-unt_score:+.3f} )",
    fontsize=14, fontweight="bold", color="white", y=0.97,
)
plt.show()

## 6. Track generalization grid

In [ ]:
from pathlib import Path
from IPython.display import Image, display

grid_path = repo_root / "results" / "track_grid.png"
if grid_path.exists():
    print(f"Track grid: {grid_path}")
    display(Image(filename=str(grid_path), width=1100))
else:
    print(f"Track grid not found at {grid_path}")
    print("Generate it with: python scripts/render_track_grid.py")

## 7. (Optional) Run GRPO training yourself

> **GPU requirements:** ≥ 20 GB VRAM recommended (Colab T4 can run the smoke version — 30 steps).
> Full 500-step run needs ≥ 32 GB (RTX 5090 / A100).

This section shows you how to reproduce the training run.

In [ ]:
# ── Check GPU availability ─────────────────────────────────────────────────
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                         "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode == 0:
    print("GPU detected:", result.stdout.strip())
else:
    print("No GPU detected — training will be CPU-only (very slow)")
    print("For GPU training, use a Colab Pro A100 runtime or your own server.")

In [ ]:
# ── Install training deps (TRL + Unsloth) ─────────────────────────────────
# Comment this out if already installed
%pip install -q "trl>=0.18.2" "peft>=0.13.2" "datasets>=3.2.0"

# For Unsloth (optional, 2x speed):
# %pip install -q "unsloth[cu128-torch260] @ https://github.com/unslothai/unsloth/archive/refs/heads/main.zip"

In [ ]:
# ── Generate SFT seed dataset ──────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ["python", "capture_everything.py",
     "--n-scenarios", "5",
     "--output", "results/sft_smoke.jsonl"],
    cwd=str(repo_root),
    capture_output=True, text=True,
)
print(result.stdout[-800:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-400:])

In [ ]:
# ── Run GRPO smoke training (30 steps, local backend, no GPU needed) ──────
# For the REAL run on a GPU server:
#   python train.py --backend trl --steps 500 --model Qwen/Qwen3-4B

import subprocess
result = subprocess.run(
    ["python", "train.py",
     "--backend", "local-smoke",
     "--steps", "20",
     "--output-dir", "grpo_colab_smoke"],
    cwd=str(repo_root),
    capture_output=True, text=True,
)
print(result.stdout[-1000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

---

## Summary

| | Random | Untrained | GRPO-trained | Expert |
|--|--|--|--|--|
| Dry sprint | ~0.30 | ~0.51 | ~0.97 | ~0.97 |
| Weather roulette | ~0.33 | ~0.38 | ~0.95 | ~0.95 |
| Late safety car | ~0.31 | ~0.42 | ~0.94 | ~0.94 |
| Championship decider | ~0.20 | ~0.28 | ~0.54 | ~0.97 |

GRPO closed **+0.46 to +0.57** of the gap to the hand-authored expert across every
scenario family, on held-out seeds the model never saw during training.

**Links:**
- HF Space: [Deltasthic/f1-strategist](https://huggingface.co/spaces/Deltasthic/f1-strategist)
- GitHub: [Deltasthicc/f1-strategist](https://github.com/Deltasthicc/f1-strategist)
- Model: [Deltasthic/f1-strategist-qwen3-4b-grpo](https://huggingface.co/Deltasthic/f1-strategist-qwen3-4b-grpo)